# VPIN Order Flow Toxicity Filter

**Filter Logic:**
- Block entries when VPIN > 0.7 (high toxicity)
- Allow entries when VPIN < 0.3 (low toxicity)
- Wraps existing strategies to reduce adverse selection

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.features.order_flow_features import vpin
from medusa.strategies import VolumeMomentumStrategy, VPINFilterStrategy

%matplotlib inline

In [ ]:
# Generate sample data with buy/sell volume
np.random.seed(42)
n_bars = 1000
dates = pd.date_range('2024-01-01', periods=n_bars, freq='5min')

price = 100 + np.cumsum(np.random.randn(n_bars) * 0.1)
total_volume = np.random.lognormal(5, 1, n_bars)

# Simulate buy/sell split (toxic periods have more imbalance)
toxicity_regime = np.random.choice([0, 1], size=n_bars, p=[0.7, 0.3])  # 30% toxic
buy_pct = np.where(toxicity_regime == 1,
                   np.random.beta(8, 2, n_bars),  # Imbalanced (toxic)
                   np.random.beta(5, 5, n_bars))  # Balanced (clean)

data = pd.DataFrame({
    'close': price,
    'high': price * (1 + abs(np.random.randn(n_bars)) * 0.005),
    'low': price * (1 - abs(np.random.randn(n_bars)) * 0.005),
    'volume': total_volume,
    'buy_volume': total_volume * buy_pct,
}, index=dates)

print(f"Data shape: {data.shape}")
data.head()

In [ ]:
# Calculate VPIN
vpin_series = vpin(data, volume_bucket_size=50, window=50)

print("\nVPIN statistics:")
print(f"  Mean: {vpin_series.mean():.3f}")
print(f"  Std: {vpin_series.std():.3f}")
print(f"  Min: {vpin_series.min():.3f}")
print(f"  Max: {vpin_series.max():.3f}")
print(f"  % High toxicity (>0.7): {(vpin_series > 0.7).mean():.1%}")
print(f"  % Low toxicity (<0.3): {(vpin_series < 0.3).mean():.1%}")

In [ ]:
# Visualize VPIN over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price
axes[0].plot(data.index, data['close'], label='Price', color='blue')
axes[0].set_ylabel('Price')
axes[0].set_title('Price Chart')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# VPIN
axes[1].plot(vpin_series.index, vpin_series, label='VPIN', color='purple', linewidth=1.5)
axes[1].axhline(0.7, color='red', linestyle='--', label='High toxicity (0.7)')
axes[1].axhline(0.3, color='green', linestyle='--', label='Low toxicity (0.3)')
axes[1].fill_between(vpin_series.index, 0, 0.3, alpha=0.2, color='green', label='Safe zone')
axes[1].fill_between(vpin_series.index, 0.7, 1.0, alpha=0.2, color='red', label='Danger zone')
axes[1].set_ylabel('VPIN')
axes[1].set_xlabel('Time')
axes[1].set_title('VPIN Order Flow Toxicity')
axes[1].set_ylim(0, 1)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare unfiltered vs VPIN-filtered signals
base_strategy = VolumeMomentumStrategy(lookback_windows=[30], volume_threshold=0.5)
filtered_strategy = VPINFilterStrategy(
    underlying_strategy=base_strategy,
    vpin_high_threshold=0.7,
    vpin_low_threshold=0.3,
)

# Generate signals
base_entries, base_exits = base_strategy.generate_signals(data)
filtered_entries, filtered_exits = filtered_strategy.generate_signals(data)

print(f"\nBase strategy entries: {base_entries.sum()}")
print(f"VPIN-filtered entries: {filtered_entries.sum()}")
print(f"Blocked by VPIN: {base_entries.sum() - filtered_entries.sum()}")
print(f"Block rate: {1 - (filtered_entries.sum() / base_entries.sum()):.1%}")

In [ ]:
# Visualize filtering effect
plt.figure(figsize=(14, 6))

# Plot price
plt.plot(data.index, data['close'], label='Price', color='blue', alpha=0.5)

# Base strategy signals
base_entry_prices = data.loc[base_entries, 'close']
plt.scatter(base_entry_prices.index, base_entry_prices, marker='^',
            color='gray', label='Base entries', s=50, alpha=0.5, zorder=3)

# Filtered strategy signals
filtered_entry_prices = data.loc[filtered_entries, 'close']
plt.scatter(filtered_entry_prices.index, filtered_entry_prices, marker='^',
            color='green', label='VPIN-filtered entries', s=100, zorder=5)

plt.ylabel('Price')
plt.xlabel('Time')
plt.title('VPIN Filter Effect: Base vs Filtered Entries')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Key Insights

VPIN filter reduces entries during toxic flow periods, potentially improving:
- Win rate (fewer entries when informed traders dominate)
- Sharpe ratio (lower drawdowns from adverse selection)
- Risk-adjusted returns

Trade-off: Lower signal frequency → fewer opportunities